In [1]:
import pandas as pd
from tqdm import tqdm

In [2]:
df_ground_truth = pd.read_csv("data/ground_truth.csv") #-data.csv")

In [3]:
df_ground_truth.shape

(1196, 2)

In [4]:
df_ground_truth.head()

,question,document
0,I’ve got about 25 minutes and some apples at h...,2
1,Can you suggest a quick applesauce recipe that...,2
2,I’ve got about an hour and a bunch of apples a...,5
3,Can you suggest a quick baked apple dessert us...,5
4,Got a few apples and butter at home — what’s a...,9


In [5]:
ground_truth = df_ground_truth.to_dict(orient="records")


In [6]:
from ingest import load_data, build_index

documents = load_data()
index = build_index(documents)

In [7]:
documents[0]

{'id': '2',
 'recipe_name': "Sarah's Homemade Applesauce",
 'total_time': '25 mins',
 'servings': '4',
 'ingredients': '4  apples - peeled, cored and chopped, ¾ cup water, ¼ cup white sugar, ½ teaspoon ground cinnamon',
 'directions': "Combine apples, water, sugar, and cinnamon in a saucepan; cover and cook over medium heat until apples are soft, about 15 to 20 minutes.\nAllow apple mixture to cool, then mash with a fork or potato masher until it is the consistency you like.\n\n\n\n\n\n\n\n\n\n\n\nPhoto by cookin'mama.\ncookin mama\n",
 'nutrition': 'Total Fat 0g 0%, Sodium 3mg 0%, Total Carbohydrate 32g 12%, Dietary Fiber 4g 13%, Total Sugars 27g, Protein 0g, Vitamin C 6mg 32%, Calcium 13mg 1%, Iron 0mg 1%, Potassium 150mg 3%'}

In [8]:
boost = {'total_time': 1.0, 'ingredients': 1.0}

index.search(
    "Any quick apple dessert or snack recipe with just apples, sugar, cinnamon, and water?",
    num_results=5,
    boost_dict=boost
)

[{'id': '60',
  'recipe_name': 'Delicious Cinnamon Baked Apples',
  'total_time': '1 hrs',
  'servings': '6',
  'ingredients': '1 teaspoon butter, 2 tablespoons brown sugar, 3 teaspoons vanilla sugar, 3 teaspoons ground cinnamon, 1 teaspoon ground nutmeg, 6 large apples - peeled, cored, and sliced, 3 ½ tablespoons water',
  'directions': 'Preheat the oven to 350 degrees F (175 degrees C). Grease a large baking dish with butter.\nMix brown sugar, vanilla sugar, cinnamon, and nutmeg in a small bowl. Layer about 1/3 of the apples in the prepared baking dish; sprinkle with 1/3 of the sugar mixture. Repeat layers twice more.\nBake in preheated oven for 30 minutes. Pour water over apples and continue baking until tender, about 15 minutes more.',
  'nutrition': 'Total Fat 1g 2%, Saturated Fat 1g 3%, Cholesterol 2mg 1%, Sodium 9mg 0%, Total Carbohydrate 37g 13%, Dietary Fiber 6g 21%, Total Sugars 27g, Protein 1g, Vitamin C 10mg 49%, Calcium 29mg 2%, Iron 0mg 2%, Potassium 240mg 5%'},
 {'id': '

In [9]:
def text_search(query):
    boost_dict = {'total_time': 1.0, 'ingredients': 1.0}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict
    )

In [10]:
def compute_relevance(q, search_function):
    doc_id = q["document"]
    results = search_function(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(int(d["id"]) == int(doc_id)))

    return relevance

In [11]:
def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)

    return relevance_total

In [12]:
relevance_total = compute_relevance_total(ground_truth, text_search)

100%|██████████| 1196/1196 [00:08<00:00, 141.33it/s]


In [13]:
def hit_rate(relevance):
    cnt = 0

    for line in relevance:
        if 1 in line:
            cnt = cnt + 1

    return cnt / len(relevance)

In [14]:
hit_rate(relevance_total)

0.7332775919732442

In [15]:
len(relevance_total)

1196

In [16]:
def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                score = 1 / (rank + 1)
                total_score = total_score + score
                break

    return total_score / len(relevance)
    

In [17]:
mrr(relevance_total)

0.555058528428094

In [18]:
def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(ground_truth, search_function)

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

In [19]:
evaluate(ground_truth, text_search)

100%|██████████| 1196/1196 [00:07<00:00, 150.20it/s]


{'hit_rate': 0.7332775919732442, 'mrr': 0.555058528428094}

In [22]:
df_validation = df_ground_truth.sample(frac=0.7, random_state=42)
df_test = df_ground_truth.drop(df_validation.index)

gt_val = df_validation.to_dict(orient="records")
gt_test = df_test.to_dict(orient="records")

In [30]:
def search_boosts(query, recipe_name_boost, ingredients_boost, nutrition_boost):
    boost_dict = {
        "recipe_name": recipe_name_boost,
        "ingredients": ingredients_boost,
        "nutrition": nutrition_boost,
    }

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

In [31]:
results = []

for recipe_name_boost in [0.5, 1.0, 2.0, 5.0]:
    for ingredients_boost in [1.0, 2.0, 5.0, 10.0]:
        for nutrition_boost in [0.5, 1.0, 3.0]:
            print(f"Evaluating recipe_name_boost={recipe_name_boost}, ingredients_boost={ingredients_boost}, nutrition_boost={nutrition_boost}...")
            result = evaluate(
                gt_val,
                lambda query, recipe_name_boost=recipe_name_boost, ingredients_boost=ingredients_boost, nutrition_boost=nutrition_boost: search_boosts(
                    query,
                    recipe_name_boost,
                    ingredients_boost,
                    nutrition_boost
                )
            )

            results.append({
                "recipe_name": recipe_name_boost,
                "ingredients": ingredients_boost,
                "nutrition": nutrition_boost,
                "hit_rate": result["hit_rate"],
                "mrr": result["mrr"],
            })

Evaluating recipe_name_boost=0.5, ingredients_boost=1.0, nutrition_boost=0.5...


100%|██████████| 837/837 [00:06<00:00, 134.57it/s]


Evaluating recipe_name_boost=0.5, ingredients_boost=1.0, nutrition_boost=1.0...


100%|██████████| 837/837 [00:05<00:00, 147.58it/s]


Evaluating recipe_name_boost=0.5, ingredients_boost=1.0, nutrition_boost=3.0...


...

100%|██████████| 837/837 [00:05<00:00, 149.70it/s]


Evaluating recipe_name_boost=5.0, ingredients_boost=10.0, nutrition_boost=1.0...


100%|██████████| 837/837 [00:05<00:00, 143.03it/s]


Evaluating recipe_name_boost=5.0, ingredients_boost=10.0, nutrition_boost=3.0...


100%|██████████| 837/837 [00:05<00:00, 147.51it/s]


In [32]:
df_results = pd.DataFrame(results)
df_results.sort_values("mrr", ascending=False).head(10)

,recipe_name,ingredients,nutrition,hit_rate,mrr
30,2.0,5.0,0.5,0.747909,0.605834
31,2.0,5.0,1.0,0.751493,0.605117
45,5.0,10.0,0.5,0.734767,0.599642
47,5.0,10.0,3.0,0.731183,0.599263
46,5.0,10.0,1.0,0.735962,0.596018
32,2.0,5.0,3.0,0.741935,0.590940
15,1.0,2.0,0.5,0.740741,0.579849
18,1.0,5.0,0.5,0.712067,0.574353
16,1.0,2.0,1.0,0.738351,0.573377
27,2.0,2.0,0.5,0.728793,0.572660


In [20]:
def text_search(query):
    boost_dict = {'recipe_name': 2.0, 'ingredients': 5.0}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict
    )

In [23]:
evaluate(gt_test, text_search)

100%|██████████| 359/359 [00:02<00:00, 150.35it/s]


{'hit_rate': 0.7186629526462396, 'mrr': 0.6113741875580317}

In [24]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [25]:
doc_idx = {}

for doc in documents:
    doc_idx[doc["id"]] = doc

In [26]:
from evaluation_utils import RAGWithUsage

In [27]:
assistant = RAGWithUsage(
    index=index,
    llm_client=openai_client,
)

In [28]:
def generate_rag_answer(rec):
    question = rec["question"]
    doc_id = rec["document"]
    original_doc = doc_idx[str(doc_id)]

    answer_llm = assistant.rag(question)
    answer_orig = original_doc

    result = {
        "question": question,
        "answer_llm": answer_llm,
        "answer_orig": answer_orig,
        "document": doc_id,
    }

    return result

In [30]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [42]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, ground_truth, generate_rag_answer)

  0%|          | 0/1196 [00:00<?, ?it/s]

In [43]:
df_results = pd.DataFrame(results)
df_results.head()

,question,answer_llm,answer_orig,document
0,I’ve got about 25 minutes and some apples at h...,Yes — you can make **Sarah’s Homemade Applesau...,"{'id': '2', 'recipe_name': 'Sarah's Homemade A...",2
1,Can you suggest a quick applesauce recipe that...,I don’t know.,"{'id': '2', 'recipe_name': 'Sarah's Homemade A...",2
2,I’ve got about an hour and a bunch of apples a...,Yes — **Easy Apple Crisp with Oat Topping** fi...,"{'id': '5', 'recipe_name': 'Easy Apple Crisp w...",5
3,Can you suggest a quick baked apple dessert us...,A good match is **Apple Cinnamon Oatmeal Muffi...,"{'id': '5', 'recipe_name': 'Easy Apple Crisp w...",5
4,Got a few apples and butter at home — what’s a...,A quick option is **Skillet Apples with Cinnam...,"{'id': '9', 'recipe_name': 'Southern Fried App...",9


In [44]:
df_results.to_csv("data/rag_results.csv", index=False)

In [45]:
assistant.total_cost()

2.4511147500000012

In [46]:
assistant.reset_usage()

In [47]:
from pydantic import BaseModel, Field
from typing import Literal

class AnswerEvaluation(BaseModel):
    reasoning: str = Field(
        description="Reasoning about the quality of the answer."
    )
    score: Literal["good", "bad"] = Field(
        description="'good' if the answer is correct and complete, 'bad' otherwise."
    )

In [48]:
aqa_judge_instructions = """
You are an expert evaluator. You will be given:
1. A question from a user
2. The original suggested recipe from a database (ground truth data)
3. An recipe suggestion generated by an AI assistant

Your task is to decide if the AI recipe suggestion fulfills the user's request.

Rules:
- The AI answer does NOT need to be word-for-word identical
- It should convey the same key information
- Extra detail is fine as long as the core answer is correct
- Mark 'bad' only if the AI answer is wrong or misses the key point

Be fair and focus on correctness, not style.
""".strip()

In [49]:
aqa_judge_prompt = """
Question:
{question}

Original Recipe (ground truth):
{answer_orig}

AI Recipe:
{answer_llm}
""".strip()

In [50]:
answers = df_results.to_dict(orient="records")

In [51]:
rec = answers[0]
rec

{'question': 'I’ve got about 25 minutes and some apples at home — what’s a super easy homemade applesauce recipe I can make with just a few pantry ingredients?',
 'answer_llm': 'Yes — you can make **Sarah’s Homemade Applesauce** in about **25 minutes**.\n\n### Ingredients\n- 4 apples, peeled, cored, and chopped\n- 3/4 cup water\n- 1/4 cup white sugar\n- 1/2 teaspoon ground cinnamon\n\n### Directions\n1. Combine the apples, water, sugar, and cinnamon in a saucepan.\n2. Cover and cook over medium heat until the apples are soft, about **15 to 20 minutes**.\n3. Let it cool a bit.\n4. Mash with a fork or potato masher until it’s as smooth or chunky as you like.\n\n### Makes\n- **4 servings**\n\nIf you want, I can also help you make a **less sugary version** using the same ingredients.',
 'answer_orig': {'id': '2',
  'recipe_name': "Sarah's Homemade Applesauce",
  'total_time': '25 mins',
  'servings': '4',
  'ingredients': '4  apples - peeled, cored and chopped, ¾ cup water, ¼ cup white sug

In [52]:
prompt = aqa_judge_prompt.format(
    question=rec["question"],
    answer_orig=rec["answer_orig"],
    answer_llm=rec["answer_llm"]
)
print(prompt)

Question:
I’ve got about 25 minutes and some apples at home — what’s a super easy homemade applesauce recipe I can make with just a few pantry ingredients?

Original Recipe (ground truth):
{'id': '2', 'recipe_name': "Sarah's Homemade Applesauce", 'total_time': '25 mins', 'servings': '4', 'ingredients': '4  apples - peeled, cored and chopped, ¾ cup water, ¼ cup white sugar, ½ teaspoon ground cinnamon', 'directions': "Combine apples, water, sugar, and cinnamon in a saucepan; cover and cook over medium heat until apples are soft, about 15 to 20 minutes.\nAllow apple mixture to cool, then mash with a fork or potato masher until it is the consistency you like.\n\n\n\n\n\n\n\n\n\n\n\nPhoto by cookin'mama.\ncookin mama\n", 'nutrition': 'Total Fat 0g 0%, Sodium 3mg 0%, Total Carbohydrate 32g 12%, Dietary Fiber 4g 13%, Total Sugars 27g, Protein 0g, Vitamin C 6mg 32%, Calcium 13mg 1%, Iron 0mg 1%, Potassium 150mg 3%'}

AI Recipe:
Yes — you can make **Sarah’s Homemade Applesauce** in about **25 m

In [53]:
from evaluation_utils import llm_structured_retry

In [54]:
eval_result, usage = llm_structured_retry(
    openai_client,
    aqa_judge_instructions,
    prompt,
    AnswerEvaluation,
)

eval_result

AnswerEvaluation(reasoning="The AI recipe matches the ground truth closely: same total time, ingredients, directions, and servings. It accurately answers the user's request for a simple homemade applesauce recipe using pantry ingredients. Extra offer about a less sugary version does not conflict with the recipe.", score='good')

In [66]:
from evaluation_utils import calc_price, calc_total_price

In [60]:
calc_price(usage)

{'input_cost': 0.00048675, 'output_cost': 0.0003015, 'total_cost': 0.00078825}

In [61]:
def evaluate_aqa(question, answer_orig, answer_llm, model="gpt-5.4-mini"):
    prompt = aqa_judge_prompt.format(
        question=question,
        answer_orig=answer_orig,
        answer_llm=answer_llm
    )

    result, usage = llm_structured_retry(
        openai_client,
        aqa_judge_instructions,
        prompt,
        AnswerEvaluation,
        model=model,
    )

    return result, usage

In [62]:
def judge_record(rec):
    eval_result, usage = evaluate_aqa(
        question=rec["question"],
        answer_orig=rec["answer_orig"],
        answer_llm=rec["answer_llm"]
    )

    result = {
        "question": rec["question"],
        "document": rec["document"],
        "score": eval_result.score,
        "reasoning": eval_result.reasoning,
    }

    return result, usage

In [63]:
judge_record(answers[0])

({'question': 'I’ve got about 25 minutes and some apples at home — what’s a super easy homemade applesauce recipe I can make with just a few pantry ingredients?',
  'document': 2,
  'score': 'good',
  'reasoning': 'The AI recipe matches the ground truth closely: same total time, ingredients, serving size, and directions for a simple applesauce recipe. It conveys the key information the user requested and adds only harmless extra wording.'},
 ResponseUsage(input_tokens=649, input_tokens_details=InputTokensDetails(cache_write_tokens=0, cached_tokens=0), output_tokens=58, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=707))

In [64]:
from concurrent.futures import ThreadPoolExecutor

with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, answers, judge_record)

  0%|          | 0/1196 [00:00<?, ?it/s]

In [65]:
evaluations = []
usages = []

for evaluation, usage in results:
    evaluations.append(evaluation)
    usages.append(usage)

In [67]:
calc_total_price(usages)


1.168401

In [68]:
df_eval = pd.DataFrame(evaluations)
df_eval.head()

,question,document,score,reasoning
0,I’ve got about 25 minutes and some apples at h...,2,good,The AI recipe matches the ground truth closely...
1,Can you suggest a quick applesauce recipe that...,2,bad,The AI response does not provide any recipe su...
2,I’ve got about an hour and a bunch of apples a...,5,good,The AI answer correctly identifies the origina...
3,Can you suggest a quick baked apple dessert us...,5,good,The user asked for a quick baked apple dessert...
4,Got a few apples and butter at home — what’s a...,9,good,The AI suggestion matches the user’s request f...


In [69]:
df_eval.score.value_counts(normalize=True)

score
good    0.789298
bad     0.210702
Name: proportion, dtype: float64

In [73]:
pd.options.display.max_colwidth = 1000

In [84]:
df_eval[df_eval["score"] == "bad"].sample()

,question,document,score,reasoning
563,Any easy low-fat drink recipe for a brunch or holiday that uses frozen cranberries and is a bit sweet?,515,bad,"The AI response does not provide any recipe or relevant suggestion, so it fails to answer the user's request for an easy low-fat sweet brunch/holiday drink using frozen cranberries."


In [86]:
df_results[df_results["document"] == 515]

,question,answer_llm,answer_orig,document
562,"What’s a quick 5-minute cocktail I can make with cranberry juice, orange juice, and champagne for one person?","A good match is **Cranberry-Pomegranate Mimosa**.\n\n- **Total time:** 5 mins \n- **Servings:** 1 \n\n**Ingredients:**\n- 1 fluid ounce cranberry-pomegranate juice\n- 1 fluid ounce orange juice\n- 2 fluid ounces champagne\n- 3 frozen cranberries\n- 2 tablespoons white sugar, or as needed\n\n**Directions:**\n1. Dip the rim of a champagne glass in water, then sugar to coat it.\n2. Put the cranberries in the bottom of the glass.\n3. Add the cranberry-pomegranate juice, orange juice, and champagne.\n\nIf you want, I can also give you a version without the sugared rim.","{'id': '515', 'recipe_name': 'Cranberry-Pomegranate Mimosa', 'total_time': '5 mins', 'servings': '1', 'ingredients': '2 tablespoons white sugar, or as needed, 3 frozen cranberries, 1 fluid ounce cranberry-pomegranate juice (such as Ocean Spray®), 1 fluid ounce orange juice, 2 fluid ounces champagne', 'directions': 'Fill a small plate with 1/4 inch of water. Pour sugar onto a second small plate, making a depth of 1/8 to 1/4 inch. Gently dip the rim of a champagne glass into the water. Raise the glass to allow excess water to drip off, then dip the rim into the sugar. Twist the glass to ensure the entire rim is coated. Discard remaining sugar. Place cranberries in the bottom of the champagne glass. Add cranberry-pomegranate juice, orange juice, and champagne.', 'nutrition': 'Total Fat 0g 0%, Sodium 8mg 0%, Total Carbohydrate 34g 13%, Dietary Fiber 0g 1%, Total Sugars 28g, Protein 0g, Vitamin C 16mg 82%, Calcium 9mg 1%, Iron 0mg 2%, Potassium 126mg 3%'}",515
563,Any easy low-fat drink recipe for a brunch or holiday that uses frozen cranberries and is a bit sweet?,I don’t know.,"{'id': '515', 'recipe_name': 'Cranberry-Pomegranate Mimosa', 'total_time': '5 mins', 'servings': '1', 'ingredients': '2 tablespoons white sugar, or as needed, 3 frozen cranberries, 1 fluid ounce cranberry-pomegranate juice (such as Ocean Spray®), 1 fluid ounce orange juice, 2 fluid ounces champagne', 'directions': 'Fill a small plate with 1/4 inch of water. Pour sugar onto a second small plate, making a depth of 1/8 to 1/4 inch. Gently dip the rim of a champagne glass into the water. Raise the glass to allow excess water to drip off, then dip the rim into the sugar. Twist the glass to ensure the entire rim is coated. Discard remaining sugar. Place cranberries in the bottom of the champagne glass. Add cranberry-pomegranate juice, orange juice, and champagne.', 'nutrition': 'Total Fat 0g 0%, Sodium 8mg 0%, Total Carbohydrate 34g 13%, Dietary Fiber 0g 1%, Total Sugars 28g, Protein 0g, Vitamin C 16mg 82%, Calcium 9mg 1%, Iron 0mg 2%, Potassium 126mg 3%'}",515


In [71]:
df_eval.to_csv("data/rag_llm_judge.csv", index=False)